In [ ]:
import os
import nibabel as nib
import numpy as np

base_dir = r"/home/shreyas/Programming_Claude/ICMR/Data"

print(f"{'Patient':<12} {'FLAIR Shape':<20} {'T1 Shape':<20} {'T2 Shape':<20} {'Aligned?'}")
print("-" * 85)

issues = []

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue

    patient_num = folder_name.split("-")[1]

    flair_path = os.path.join(root, f"{patient_num}-Flair.nii")
    t1_path    = os.path.join(root, f"{patient_num}-T1.nii")
    t2_path    = os.path.join(root, f"{patient_num}-T2.nii")
    label_path = os.path.join(root, f"{patient_num}-LesionSeg-Flair.nii")

    paths = {"FLAIR": flair_path, "T1": t1_path, "T2": t2_path, "Label": label_path}
    missing = [k for k, v in paths.items() if not os.path.exists(v)]

    if missing:
        print(f"Patient-{patient_num:<6} MISSING: {missing}")
        issues.append((patient_num, f"missing {missing}"))
        continue

    flair = nib.load(flair_path).get_fdata()
    t1    = nib.load(t1_path).get_fdata()
    t2    = nib.load(t2_path).get_fdata()

    aligned = (flair.shape == t1.shape == t2.shape)
    status  = "✓" if aligned else "✗ MISMATCH"

    print(f"Patient-{patient_num:<6} {str(flair.shape):<20} {str(t1.shape):<20} {str(t2.shape):<20} {status}")

    if not aligned:
        issues.append((patient_num, f"shape mismatch: FLAIR{flair.shape} T1{t1.shape} T2{t2.shape}"))

print(f"\n--- Summary ---")
print(f"Total issues : {len(issues)}")
for p, reason in issues:
    print(f"  Patient-{p}: {reason}")

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
                "segmentation-models-pytorch", "albumentations"], check=True)

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from monai.losses import DiceLoss

print("All imports successful.")

In [ ]:

base_dir = r""

image_slices = []   
prev_slices  = []   
next_slices  = []   
label_slices = []   

patients_loaded = 0

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue

    patient_num = folder_name.split("-")[1]
    flair_path  = os.path.join(root, f"{patient_num}-Flair.nii")
    label_path  = os.path.join(root, f"{patient_num}-LesionSeg-Flair.nii")

    if not (os.path.exists(flair_path) and os.path.exists(label_path)):
        continue

    
    flair_vol = nib.load(flair_path).get_fdata().astype(np.float32)
    label_vol = nib.load(label_path).get_fdata().astype(np.float32)

    
    flair_vol = np.clip(flair_vol, 0, 1000) / 1000.0

    
    label_vol = (label_vol > 0).astype(np.float32)

    num_slices = flair_vol.shape[2]

    for z in range(num_slices):
        
        curr = flair_vol[:, :, z]

        
        prev = flair_vol[:, :, z - 1] if z > 0 \
               else np.zeros_like(curr)

        
        nxt  = flair_vol[:, :, z + 1] if z < num_slices - 1 \
               else np.zeros_like(curr)

        image_slices.append(curr)
        prev_slices.append(prev)
        next_slices.append(nxt)
        label_slices.append(label_vol[:, :, z])

    patients_loaded += 1

print(f"Patients loaded       : {patients_loaded}")
print(f"Total 2D slices       : {len(image_slices)}")
print(f"Sample slice shape    : {image_slices[0].shape}")


total_pixels  = sum(l.size for l in label_slices)
lesion_pixels = sum((l > 0).sum() for l in label_slices)
empty_slices  = sum(1 for l in label_slices if l.max() == 0)
print(f"Lesion pixel ratio    : {100 * lesion_pixels / total_pixels:.4f}%")
print(f"Empty slices (no lesion): {empty_slices}/{len(label_slices)}")

In [ ]:

TARGET_SIZE = 256

train_aug = A.Compose([
    A.Resize(TARGET_SIZE, TARGET_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1,
                       rotate_limit=15, p=0.5),
    A.ElasticTransform(p=0.3),
    A.GaussianBlur(p=0.2),
    A.RandomBrightnessContrast(p=0.3),
], additional_targets={"prev": "image", "nxt": "image"})

val_aug = A.Compose([
    A.Resize(TARGET_SIZE, TARGET_SIZE),
], additional_targets={"prev": "image", "nxt": "image"})


class SliceDataset25D(Dataset):
    """
    Returns 3-channel input: [prev_slice, curr_slice, next_slice]
    This gives the model 2.5D spatial context without requiring
    full 3D volumes — critical for thin-slice MRI datasets.
    """
    def __init__(self, curr, prev, nxt, labels, transform=None):
        self.curr      = curr
        self.prev      = prev
        self.nxt       = nxt
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.curr)

    def __getitem__(self, idx):
        curr  = self.curr[idx]
        prev  = self.prev[idx]
        nxt   = self.nxt[idx]
        label = self.labels[idx]

        if self.transform:
            aug   = self.transform(image=curr, prev=prev,
                                   nxt=nxt, mask=label)
            curr  = aug["image"]
            prev  = aug["prev"]
            nxt   = aug["nxt"]
            label = aug["mask"]

        
        curr  = torch.tensor(curr,  dtype=torch.float32)
        prev  = torch.tensor(prev,  dtype=torch.float32)
        nxt   = torch.tensor(nxt,   dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)

        
        if curr.dim() == 3:
            curr  = curr[0]
        if prev.dim() == 3:
            prev  = prev[0]
        if nxt.dim() == 3:
            nxt   = nxt[0]

        image = torch.stack([prev, curr, nxt], dim=0)  

        return image, label

print("Dataset class defined.")

In [ ]:
slice_counts = []
patient_nums = []

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue
    patient_num = folder_name.split("-")[1]
    flair_path  = os.path.join(root, f"{patient_num}-Flair.nii")
    if os.path.exists(flair_path):
        vol = nib.load(flair_path).get_fdata()
        slice_counts.append(vol.shape[2])
        patient_nums.append(int(patient_num))


paired       = sorted(zip(patient_nums, slice_counts))
slice_counts = [s for _, s in paired]

total_patients  = len(slice_counts)
split_idx       = int(0.8 * total_patients)       
train_slice_end = sum(slice_counts[:split_idx])

train_curr   = image_slices[:train_slice_end]
train_prev   = prev_slices[:train_slice_end]
train_nxt    = next_slices[:train_slice_end]
train_labels = label_slices[:train_slice_end]

val_curr     = image_slices[train_slice_end:]
val_prev     = prev_slices[train_slice_end:]
val_nxt      = next_slices[train_slice_end:]
val_labels   = label_slices[train_slice_end:]


import random
pos_slices = []
neg_slices = []
for c, p, n, l in zip(train_curr, train_prev, train_nxt, train_labels):
    if l.max() > 0:
        pos_slices.append((c, p, n, l))
    else:
        neg_slices.append((c, p, n, l))


random.seed(42)
sampled_neg = random.sample(neg_slices, min(len(pos_slices), len(neg_slices)))
filtered = pos_slices + sampled_neg
random.shuffle(filtered)

train_curr, train_prev, train_nxt, train_labels = map(list, zip(*filtered))
print(f"Train patients        : {split_idx}")
print(f"Val patients          : {total_patients - split_idx}")
print(f"Train slices (with lesion) : {len(train_curr)}")
print(f"Val slices (all)           : {len(val_curr)}")

train_ds = SliceDataset25D(train_curr, train_prev, train_nxt,
                           train_labels, transform=train_aug)
val_ds   = SliceDataset25D(val_curr,   val_prev,   val_nxt,
                           val_labels,  transform=val_aug)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"Train batches         : {len(train_loader)}")
print(f"Val batches           : {len(val_loader)}")

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")
if device.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")


model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b4",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
).to(device)


dice_loss_fn = smp.losses.DiceLoss(mode="binary")
focal_loss_fn = smp.losses.FocalLoss(mode="binary")

def combined_loss(outputs, labels):
    return 0.5 * focal_loss_fn(outputs, labels) + \
           0.5 * dice_loss_fn(outputs, labels)

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-6, verbose=True
)

print(f"Trainable parameters : "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Model, loss, optimizer ready.")

In [ ]:
def dice_coefficient(pred, target, threshold=0.5, smooth=1e-6):
    pred  = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum()
    return ((2 * inter + smooth) /
            (pred.sum() + target.sum() + smooth)).item()

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

RESUME_FROM   = None    
max_epochs    = 150
best_val_dice = 0.0
best_epoch    = 0
train_loss_log = []
val_dice_log   = []
start_epoch   = 1
PATIENCE      = 20
no_improve    = 0
ACCUM_STEPS   = 2       

if RESUME_FROM and os.path.exists(RESUME_FROM):
    checkpoint = torch.load(RESUME_FROM, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        best_val_dice  = checkpoint["best_val_dice"]
        train_loss_log = checkpoint.get("train_loss_log", [])
        val_dice_log   = checkpoint.get("val_dice_log", [])
        start_epoch    = checkpoint["epoch"] + 1
        print(f"Resumed from epoch {checkpoint['epoch']} | "
              f"Best Dice: {best_val_dice:.4f}")
    else:
        model.load_state_dict(checkpoint)
        print(f"Loaded weights from {RESUME_FROM}")
else:
    print("Starting training from scratch.")

for epoch in range(start_epoch, max_epochs + 1):

    
    model.train()
    epoch_loss  = 0.0
    epoch_steps = 0
    optimizer.zero_grad()

    with tqdm(train_loader,
              desc=f"Epoch {epoch:>3}/{max_epochs} [Train]",
              leave=False) as pbar:
        for step, (images_b, labels_b) in enumerate(pbar):
            images_b = images_b.to(device)
            labels_b = labels_b.unsqueeze(1).to(device)  

            outputs = model(images_b)
            loss    = combined_loss(outputs, labels_b) / ACCUM_STEPS
            loss.backward()

            if (step + 1) % ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                torch.cuda.empty_cache()

            epoch_loss  += loss.item() * ACCUM_STEPS
            epoch_steps += 1
            pbar.set_postfix(loss=f"{loss.item() * ACCUM_STEPS:.4f}")

        
        optimizer.step()
        optimizer.zero_grad()
        torch.cuda.empty_cache()

    avg_train_loss = epoch_loss / epoch_steps
    train_loss_log.append(avg_train_loss)

   
    if epoch % 2 == 0:
        model.eval()
        torch.cuda.empty_cache()
        val_dices = []

        with torch.no_grad():
            for images_b, labels_b in val_loader:
                images_b = images_b.to(device)
                labels_b = labels_b.unsqueeze(1).to(device)
                outputs  = model(images_b)
                dice     = dice_coefficient(outputs, labels_b)
                val_dices.append(dice)

        mean_dice = np.mean(val_dices)
        val_dice_log.append((epoch, mean_dice))
        scheduler.step(mean_dice)

        print(f"Epoch [{epoch:>3}/{max_epochs}]  "
              f"Train Loss: {avg_train_loss:.4f}  |  "
              f"Val Dice: {mean_dice:.4f}")

        if mean_dice > best_val_dice:
            best_val_dice = mean_dice
            best_epoch    = epoch
            no_improve    = 0
            torch.save(model.state_dict(),
                       "best_unetpp_efficientnet.pth")
            print(f"  ✓ New best model saved (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1
            print(f"  No improvement for {no_improve} val checks "
                  f"(patience: {PATIENCE})")
            if no_improve >= PATIENCE:
                print(f"\n⚠ Early stopping at epoch {epoch}")
                break
    else:
        print(f"Epoch [{epoch:>3}/{max_epochs}]  "
              f"Train Loss: {avg_train_loss:.4f}")

    
    if epoch % 10 == 0:
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_dice":        best_val_dice,
            "train_loss_log":       train_loss_log,
            "val_dice_log":         val_dice_log,
        }, f"checkpoint_unetpp_epoch_{epoch}.pth")
        print(f"  💾 Checkpoint saved at epoch {epoch}")

    torch.cuda.empty_cache()

print(f"\nTraining complete.")
print(f"Best Val Dice : {best_val_dice:.4f} at epoch {best_epoch}")

In [ ]:

import matplotlib.pyplot as plt

epochs_logged = list(range(1, len(train_loss_log) + 1))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_logged, train_loss_log, color="steelblue", linewidth=1)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

if val_dice_log:
    val_epochs = [x[0] for x in val_dice_log]
    val_dices  = [x[1] for x in val_dice_log]
    axes[1].plot(val_epochs, val_dices, color="darkorange",
                 linewidth=1, marker="o", markersize=3)
    axes[1].set_title("Validation Dice Score")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves_unetpp.png", dpi=150)
plt.show()

print(f"Total epochs          : {len(train_loss_log)}")
print(f"Best val Dice         : {max(val_dice_log, key=lambda x: x[1])}")
print(f"Worst val Dice        : {min(val_dice_log, key=lambda x: x[1])}")
print(f"Latest val Dice       : {val_dice_log[-1][1]:.4f}")
print(f"Final train loss      : {train_loss_log[-1]:.4f}")
print(f"Best train loss       : {min(train_loss_log):.4f}")
print(f"Current LR            : {optimizer.param_groups[0]['lr']}")


model.eval()
all_preds = []
with torch.no_grad():
    for images_b, labels_b in val_loader:
        images_b = images_b.to(device)
        outputs  = torch.sigmoid(model(images_b))
        preds    = (outputs > 0.5).float()
        all_preds.append(preds.sum().item())

total_lesion_preds = sum(all_preds)
print(f"\nTotal lesion pixels predicted in val set : {total_lesion_preds:,.0f}")
print(f"(0 means model predicts all background)")